In [25]:
import os
import requests
from dotenv import load_dotenv
from getpass import getpass
from datetime import datetime
from langfuse import get_client, Evaluation
from qdrant_client.http.models import SearchParams
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode

In [26]:
COLLECTION_NAME = "20260520_best_coll_with_urls"
AVG_LEN_SPARSE_FR = 254.77109004739336 # TODO : avg_len a recuperer autrement
K = 100

load_dotenv()
langfuse = get_client()

if not os.getenv("BGE_API_KEY_EMBEDDINGS"):
    os.environ["BGE_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model (bge): ")

DATASET_NAME = "20250520_clean_dev_dataset"
RUN_NAME = datetime.now().isoformat()

dataset = langfuse.get_dataset(DATASET_NAME)

In [27]:
# reranker part

if not os.getenv("BGE_API_KEY_RERANKER"):
    os.environ["BGE_API_KEY_RERANKER"] = getpass("Enter your RCP API key for reranking model (bge): ")

api_key = os.environ["BGE_API_KEY_RERANKER"]
base_url = 'https://inference.rcp.epfl.ch/v1'

In [28]:
embeddings = OpenAIEmbeddings(
    model="BAAI/bge-m3",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("BGE_API_KEY_EMBEDDINGS")
)

sparse = FastEmbedSparse(model_name="Qdrant/bm25", avg_len= AVG_LEN_SPARSE_FR, language="french")

qdrant = QdrantVectorStore.from_existing_collection(
    url="http://localhost:6333",
    embedding=embeddings,
    sparse_embedding=sparse,
    collection_name=COLLECTION_NAME,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse_fr"
)

In [29]:
# source : https://gitlab.epfl.ch/rcp/aiaas/client-usage-examples/-/blob/main/requests-reranker.py

# TODO : top_n doit etre une constante qq part
def rerank_documents(api_key, base_url, query, documents, model="BAAI/bge-reranker-v2-m3", top_n=20):
    """
    Reranks documents based on their relevance to a given query.

    Args:
        api_key (str): Your API key
        base_url (str): The base URL for the reranking API
        query (str): The query to rerank documents for
        documents (list[str]): A list of documents to rerank
        model (str, optional): The model to use for reranking. Defaults to "BAAI/bge-reranker-v2-m3".
        top_n (int, optional): The number of top documents to return. Defaults to 3.

    Returns:
        list[str]: The reranked documents
    """
    url = f"{base_url}/rerank"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "query": query,
        "documents": documents,
        "top_n": top_n
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    result = response.json()

    ranked_documents = sorted(result["results"], key=lambda x: x["relevance_score"], reverse=True)

    return ranked_documents

In [30]:
def make_retrieval_task(api_key, base_url):
    def retrieval_task(*, item, **kwargs):
        query = item.input.get("query")
        hits = qdrant.similarity_search_with_score(query, k=K, search_params=SearchParams(exact=True))

        # reranker
        documents = [document.page_content for document, _ in hits]
        reranked_hits = rerank_documents(api_key, base_url, query, documents)

        retrieved_doc_ids = []
        reranked_scores = []

        for hit in reranked_hits:
            original_index = hit["index"]
            rerank_score = hit["relevance_score"]
            document, _ = hits[original_index]
            doc_id = document.metadata.get("doc_id")
            retrieved_doc_ids.append(doc_id)
            reranked_scores.append(rerank_score)

        return {
            "retrieved_doc_ids": retrieved_doc_ids,
            "retrieved_scores": reranked_scores
        }
    return retrieval_task

In [31]:
def make_hit_at_x_evaluator(x):
    def hit_at_x_evaluator(*, output, metadata, **kwargs):
        expected_doc_id = metadata.get("expected_doc_id")
        retrieved_doc_ids_top_k = output.get("retrieved_doc_ids")[:x]
        retrieved_scores_top_k = output.get("retrieved_scores")[:x]
        value = 1.0 if expected_doc_id in retrieved_doc_ids_top_k else 0.0
        return Evaluation(
            name=f"hit_at_{x}",
            value=value,
            comment=f"expected_doc_id={expected_doc_id}, top{x}={retrieved_doc_ids_top_k} with score {retrieved_scores_top_k}"
        )
    return hit_at_x_evaluator

def mrr_doc_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    reciprocal_rank = 0.0
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id == expected_doc_id:
            reciprocal_rank = 1.0 / rank
            break
    return Evaluation(
        name="mrr_doc",
        value=reciprocal_rank,
        comment=f"expected_doc_id={expected_doc_id} and retrieved={retrieved_doc_ids}",
    )

def nb_correct_doc_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    nb_correct_docs = 0
    for retrieved_doc_id in retrieved_doc_ids:
        if retrieved_doc_id == expected_doc_id:
            nb_correct_docs += 1
    return Evaluation(
        name="nb_correct_doc",
        value=nb_correct_docs,
        comment=f"expected_doc_id={expected_doc_id} and retrieved={retrieved_doc_ids}",
    )

In [32]:
result = dataset.run_experiment(
    name=COLLECTION_NAME,
    run_name=RUN_NAME,
    description=f"Reranking experiment to score retrieval task on {DATASET_NAME} dataset using {COLLECTION_NAME} collection",
    task=make_retrieval_task(api_key, base_url),
    evaluators=[
        make_hit_at_x_evaluator(1),
        make_hit_at_x_evaluator(5),
        make_hit_at_x_evaluator(20),
        mrr_doc_evaluator,
        nb_correct_doc_evaluator
    ],
    metadata={
        "collection_name": COLLECTION_NAME,
        "top_k": K,
    }
)

print(result.format())

Individual Results: Hidden (13 items)\n💡 Set include_item_results=True to view them\n\n──────────────────────────────────────────────────\n🧪 Experiment: 20260520_best_coll_with_urls
📋 Run name: 2026-05-20T16:47:18.914894 - Reranking experiment to score retrieval task on 20250520_clean_dev_dataset dataset using 20260520_best_coll_with_urls collection\n13 items\nEvaluations:\n  • mrr_doc\n  • hit_at_20\n  • hit_at_1\n  • hit_at_5\n  • nb_correct_doc\n\nAverage Scores:\n  • mrr_doc: 0.631\n  • hit_at_20: 0.769\n  • hit_at_1: 0.538\n  • hit_at_5: 0.769\n  • nb_correct_doc: 2.385\n\n🔗 Dataset Run:\n   http://localhost:3000/project/cmmvqmjqn0007nw07hbqsche1/datasets/cmpdor8yr0026nw07btz59oao/runs/310b3b7f-d365-4374-abfd-499698700568
